In [0]:
"""
Push DQ Results to OpenMetadata
Reads _summary_columns / _summary_dataset from Databricks Delta tables
and pushes test cases + results to OpenMetadata via its REST API.

Triggered by the backend (via jobs.runs/submit) using a job cluster
so the cluster is on the internal network and can reach OMD intranet.
"""
import json
import os
import time
import requests
import urllib3
from datetime import datetime
from urllib.parse import quote

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ---------------------------------------------------------------------------
# Parameters
# ---------------------------------------------------------------------------
dbutils.widgets.text("norm_project", "")
dbutils.widgets.text("norm_job", "")
dbutils.widgets.text("omd_fqn", "")
dbutils.widgets.text("omd_base_url", "")
dbutils.widgets.text("databricks_host", "")
dbutils.widgets.text("databricks_http_path", "")
dbutils.widgets.text("payload_json", "")
dbutils.widgets.text("payload_path", "")

NORM_PROJECT = dbutils.widgets.get("norm_project")
NORM_JOB     = dbutils.widgets.get("norm_job")
_raw_fqn     = dbutils.widgets.get("omd_fqn").strip()
OMD_FQN      = (" " + _raw_fqn) if _raw_fqn.startswith("starburst-uat01") else _raw_fqn
# Token read from environment variable — never passed as a task parameter
OMD_TOKEN    = os.getenv("OMD_TOKEN", "")
OMD_BASE_URL = dbutils.widgets.get("omd_base_url").rstrip("/")
DATABRICKS_HOST = dbutils.widgets.get("databricks_host")
DATABRICKS_HTTP_PATH = dbutils.widgets.get("databricks_http_path")
PAYLOAD_JSON = None
PAYLOAD_PATH = dbutils.widgets.get("payload_path")

try:
    PAYLOAD_JSON = dbutils.jobs.taskValues.get("prepare_payload", "payload_json")
    print("  → Loaded payload_json from upstream task values")
except Exception as exc:
    print(f"  → Upstream task values unavailable: {exc}")
    PAYLOAD_JSON = dbutils.widgets.get("payload_json")

if not PAYLOAD_JSON and PAYLOAD_PATH:
    try:
        PAYLOAD_JSON = dbutils.fs.head(PAYLOAD_PATH)
        print(f"  → Loaded payload_json from DBFS path: {PAYLOAD_PATH}")
    except Exception as exc:
        print(f"  → Failed to read payload from DBFS path: {exc}")

print(f"=== Push DQ to OpenMetadata ===")
print(f"Project : {NORM_PROJECT}")
print(f"Job     : {NORM_JOB}")
print(f"FQN     : {OMD_FQN}")
print(f"OMD URL : {OMD_BASE_URL}")
print(f"Token   : {'SET' if OMD_TOKEN else 'MISSING'}")

if not OMD_TOKEN:
    raise ValueError("omd_token parameter is empty — cannot authenticate with OpenMetadata")
if not OMD_FQN:
    raise ValueError("omd_fqn parameter is empty")

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def _headers():
    return {
        "Authorization": f"Bearer {OMD_TOKEN}",
        "Accept": "application/json",
        "Content-Type": "application/json",
    }

def safe_float(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return None


def _req(method, url, **kwargs):
    """Wrapper around requests with 1 automatic retry on ConnectionError."""
    for attempt in range(2):
        try:
            return getattr(requests, method)(url, **kwargs)
        except requests.exceptions.ConnectionError as exc:
            if attempt == 0:
                print(f"  ⚠ ConnectionError on {url} — retrying in 3s…")
                time.sleep(3)
            else:
                raise exc

def _load_summary_rows():
    """Load rows from the payload passed as a task parameter."""
    if not PAYLOAD_JSON:
        raise ValueError("payload_json parameter is missing — the first task must pass it")

    try:
        payload = json.loads(PAYLOAD_JSON)
    except json.JSONDecodeError as exc:
        raise ValueError(f"payload_json is not valid JSON: {exc}") from exc

    columns_rows = payload.get("columns_rows", [])
    dataset_rows = payload.get("dataset_rows", [])
    if not columns_rows and not dataset_rows:
        raise ValueError("payload_json did not contain any DQ rows")

    print("  → Loaded DQ rows from the payload passed by the first task")
    return columns_rows, dataset_rows

# ---------------------------------------------------------------------------
# 1. Read latest run from Delta tables via Databricks SQL
# ---------------------------------------------------------------------------
print(f"\n[1/4] Reading DQ results...")
columns_rows, dataset_rows = _load_summary_rows()
print(f"  → {len(columns_rows)} column rows, {len(dataset_rows)} dataset rows")

if not columns_rows:
    raise ValueError("No DQ results found in the payload — the preparation notebook did not provide any rows")

# ---------------------------------------------------------------------------
# 2. Ensure Test Suite
# ---------------------------------------------------------------------------
print(f"\n[2/4] Ensuring test suite for asset: {OMD_FQN}")

ts_body = {
    "name": OMD_FQN + ".testSuite",
    "executableEntityReference": OMD_FQN,
    "basicEntityReference": OMD_FQN,
}
ts_resp = _req(
    "put",
    f"{OMD_BASE_URL}/v1/dataQuality/testSuites/executable",
    headers=_headers(), json=ts_body, verify=False
)
print(f"  → Test suite PUT {ts_resp.status_code}")

# ---------------------------------------------------------------------------
# 3. Table-level test case + result
# ---------------------------------------------------------------------------
print(f"\n[3/4] Pushing table-level test case...")

table_name     = OMD_FQN.split(".")[-1]
tc_name        = table_name + " dq summary"
entity_link    = f"<#E::table::{OMD_FQN}>"
unix_timestamp = int(datetime.now().timestamp() * 1000)

tc_body = {
    "description": f"Test_Case_{table_name}",
    "displayName": tc_name,
    "entityLink": entity_link,
    "name": tc_name,
    "parameterValues": [
        {"name": "minValue", "value": 0},
        {"name": "maxValue", "value": 10000000},
    ],
    "testDefinition": "tableRowCountToBeBetween",
}
tc_resp = _req(
    "post",
    f"{OMD_BASE_URL}/v1/dataQuality/testCases",
    headers=_headers(), json=tc_body, verify=False
)
print(f"  → Table test case POST {tc_resp.status_code}")

if tc_resp.status_code in (200, 201):
    tc_fqn = tc_resp.json().get("fullyQualifiedName", f"{OMD_FQN}.{tc_name}")
elif tc_resp.status_code == 409:
    tc_fqn = f"{OMD_FQN}.{tc_name}"
else:
    print(f"  ⚠ Table test case creation failed ({tc_resp.status_code}): {tc_resp.text}")
    tc_fqn = None

if tc_fqn:
    scores = {r["check_type"]: safe_float(r["dataset_score"]) for r in dataset_rows}
    valid_scores = [v for v in scores.values() if v is not None]
    avg_score = round(sum(valid_scores) / len(valid_scores), 2) if valid_scores else 0.0

    if avg_score >= 98:
        tc_status = "Success"
    elif avg_score >= 95:
        tc_status = "Aborted"
    else:
        tc_status = "Failed"

    result_values = [
        {"name": dim.capitalize(), "value": scores[dim]}
        for dim in ("completeness", "validity", "uniqueness", "consistency")
        if scores.get(dim) is not None
    ]

    result_resp = _req(
        "put",
        f"{OMD_BASE_URL}/v1/dataQuality/testCases/testCaseResults/{quote(tc_fqn, safe='')}",
        headers=_headers(),
        json={"testCaseStatus": tc_status, "testResultValue": result_values, "timestamp": unix_timestamp},
        verify=False
    )
    print(f"  → Table result PUT {result_resp.status_code} (score={avg_score}, status={tc_status})")

# ---------------------------------------------------------------------------
# 4. Column-level test cases + results
# ---------------------------------------------------------------------------
print(f"\n[4/4] Pushing column-level test cases...")

# Fetch columns registered in OMD
omd_cols_resp = _req(
    "get",
    f"{OMD_BASE_URL}/v1/tables/name/{OMD_FQN}",
    headers=_headers(), verify=False
)
omd_columns = set()
if omd_cols_resp.status_code == 200:
    omd_columns = {c["name"] for c in omd_cols_resp.json().get("columns", [])}
    print(f"  → {len(omd_columns)} columns registered in OMD")
else:
    print(f"  ⚠ Could not fetch OMD columns ({omd_cols_resp.status_code}) — skipping column filter")

# Group rows by column
by_column = {}
for row in columns_rows:
    col        = row["column_name"]
    check_type = row["check_type"]
    if col not in by_column:
        by_column[col] = {"scores": {}, "statuses": {}, "faulty": {}, "overall_status": None}
    if check_type == "overall":
        by_column[col]["overall_status"] = row.get("status")
    else:
        by_column[col]["scores"][check_type]   = row.get("score")
        by_column[col]["statuses"][check_type] = row.get("status")
        by_column[col]["faulty"][check_type]   = row.get("faulty_count")

pushed = 0
skipped = 0

for col, data in by_column.items():
    if omd_columns and col not in omd_columns:
        skipped += 1
        continue

    try:
        col_entity_link = f"<#E::table::{OMD_FQN}::columns::{col}>"

        faulty_parts = [
            f"{dim.capitalize()}Faulty: {int(data['faulty'][dim])}"
            for dim in ("completeness", "validity", "uniqueness", "consistency")
            if data["faulty"].get(dim) is not None
        ]
        description_faulty = ", ".join(faulty_parts)

        # Create/update test case for column
        col_tc_body = {
            "description": description_faulty,
            "displayName": col,
            "entityLink": col_entity_link,
            "name": col,
            "parameterValues": [
                {"name": "minLength", "value": 80},
                {"name": "maxLength", "value": 100},
            ],
            "testDefinition": "columnValueLengthsToBeBetween",
        }
        put_resp = _req(
            "put",
            f"{OMD_BASE_URL}/v1/dataQuality/testCases",
            headers=_headers(), json=col_tc_body, verify=False
        )

        # Resolve FQN via search
        search_resp = _req(
            "get",
            f"{OMD_BASE_URL}/v1/dataQuality/testCases",
            headers=_headers(),
            params={"entityLink": col_entity_link},
            verify=False
        )
        col_tc_fqn = None
        col_tc_id  = None
        if search_resp.status_code == 200:
            tc_data = search_resp.json().get("data", [])
            if tc_data:
                col_tc_id  = tc_data[0]["id"]
                col_tc_fqn = tc_data[0].get("fullyQualifiedName")

        if not col_tc_fqn:
            print(f"  ⚠ No FQN for column '{col}', skipping")
            continue

        # PATCH description
        if col_tc_id and description_faulty:
            patch_headers = _headers()
            patch_headers["Content-Type"] = "application/json-patch+json"
            patch_resp = _req(
                "patch",
                f"{OMD_BASE_URL}/v1/dataQuality/testCases/{col_tc_id}",
                headers=patch_headers,
                data=json.dumps([{"op": "replace", "path": "/description", "value": description_faulty}]),
                verify=False
            )
            if patch_resp.status_code == 200:
                col_tc_fqn = patch_resp.json().get("fullyQualifiedName", col_tc_fqn)
            time.sleep(1)

        # Push result
        overall_status = data.get("overall_status") or "KO"
        col_tc_status  = "Success" if overall_status == "OK" else "Failed"

        result_values = []
        for dim in ("completeness", "validity", "uniqueness", "consistency"):
            score = data["scores"].get(dim)
            if score is not None:
                try:
                    result_values.append({"name": dim.capitalize(), "value": float(score)})
                except (TypeError, ValueError):
                    pass

        col_result_resp = _req(
            "post",
            f"{OMD_BASE_URL}/v1/dataQuality/testCases/testCaseResults/{quote(col_tc_fqn, safe='')}",
            headers=_headers(),
            json={
                "testCaseStatus": col_tc_status,
                "testResultValue": result_values,
                "timestamp": unix_timestamp,
                "fqn": col_tc_fqn,
            },
            verify=False
        )

        if col_result_resp.status_code in (200, 201):
            pushed += 1
        else:
            print(f"  ⚠ Column '{col}' result failed ({col_result_resp.status_code}): {col_result_resp.text}")

    except Exception as col_exc:
        print(f"  ⚠ Column '{col}' skipped due to error: {col_exc}")
        continue

print(f"\n✅ Publish complete — {pushed} column(s) pushed, {skipped} skipped (not in OMD)")

# Return summary as notebook output (readable by get_run_output)
dbutils.notebook.exit(json.dumps({
    "status": "success",
    "columns_pushed": pushed,
    "columns_skipped": skipped,
    "fqn": OMD_FQN
}))
